# TP 2 : déposer, inspecter et récupérer des données dans HDFS

**Module Big Data (Hadoop & Spark), Bac+5, Institut GoldenCollar. Mouaad AGOURRAM.** Durée : 55 minutes.

## Contexte

Le cluster du module tourne dans Docker sur votre machine : un maître (`master`), trois travailleurs
(`worker1` à `worker3`) et un poste client (`client`). Ce notebook tourne hors du cluster, dans votre
environnement Python, et pilote HDFS par le conteneur client. Vous travaillez sur deux jeux de données
réels et publics :

- trois mois de trajets de taxis jaunes de New York (janvier à mars 2024), un fichier Parquet par mois
  (`yellow_tripdata_2024-01.parquet` à `-03`), téléchargés depuis `d37ci6vzurychx.cloudfront.net/trip-data/` :
  environ 50 Mo par fichier. Pour ce TP, un fichier est une suite d'octets : vous ne lisez pas son contenu ;
- les logs HTTP du serveur web du Kennedy Space Center (NASA) d'août 1995, déjà vus : une ligne par
  requête reçue. La page de description (https://ita.ee.lbl.gov/html/contrib/NASA-HTTP.html) demande de ne
  pas les analyser au-delà des tendances générales de trafic : on ne fait ici que les déposer et les compter.

## Objectifs

1. Organiser une arborescence HDFS et y déposer des fichiers avec une taille de bloc imposée.
2. Prévoir, puis vérifier, le nombre de blocs, la taille du dernier bloc et l'espace consommé.
3. Changer la réplication, localiser les copies, récupérer un fichier et en vérifier le contenu.
4. Arrêter un DataNode, observer ce que HDFS répare, et le relancer.
5. Mesurer le coût des petits fichiers pour le NameNode.

## Consignes

- Environnement : depuis la racine du module, `uv sync --group tp`, puis choisissez `.venv` comme noyau de
  ce notebook. Ouvrez-le depuis le dossier `seance2/tp`. Le cluster doit tourner (`docker compose up -d`)
  avec ses trois DataNodes.
- La fonction `cluster(commande)`, définie dans la préparation, exécute une commande shell dans le conteneur
  client, où est installé `hdfs`, et renvoie la liste des lignes de sa sortie. La fonction
  `docker_compose(*arguments)` lance une commande `docker compose` sur votre machine.
- Le dossier du module est monté dans le conteneur client sous `/cours` : le dossier `seance2/tp/data` de
  votre machine y est `/cours/seance2/tp/data`.
- Chaque question se termine par un livrable : affectez la valeur demandée à la variable indiquée.
- Le NameNode est sur `localhost:9870` : onglet Datanodes, et Utilities > Browse the file system.
- Les questions 12 et 13 sont des bonus.

## Préparation

La cellule suivante définit `cluster` et `docker_compose`, télécharge dans `data/` les trois fichiers
Parquet (160 Mo) et les logs d'août 1995 (16 Mo compressés), puis découpe les logs d'août en un fichier
par jour dans `data/jours_aout/`. Exécutez-la sans la modifier.

In [ ]:
import gzip
import hashlib
import json
import math
import re
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

COMPOSE = Path.cwd().parents[1] / "docker-compose.yml"
assert COMPOSE.exists(), f"{COMPOSE} introuvable : ouvrez ce notebook depuis le dossier seance2/tp"
CLIENT = "/cours/seance2/tp"          # le dossier seance2/tp, vu depuis le conteneur client


def cluster(commande):
    # Exécute une commande shell dans le conteneur client du cluster ; renvoie ses lignes de sortie
    resultat = subprocess.run(
        ["docker", "compose", "-f", str(COMPOSE), "exec", "-T", "client", "bash", "-c", commande],
        capture_output=True, text=True)
    if resultat.returncode != 0:
        raise RuntimeError(f"Échec de : {commande}\n{resultat.stderr}")
    return resultat.stdout.splitlines()


def docker_compose(*arguments):
    # Lance une commande docker compose sur votre machine (par exemple : docker_compose("ps"))
    resultat = subprocess.run(["docker", "compose", "-f", str(COMPOSE), *arguments], capture_output=True, text=True)
    if resultat.returncode != 0:
        raise RuntimeError(f"Échec de docker compose {' '.join(arguments)}\n{resultat.stderr}")
    return resultat.stdout


DATA = Path("data")
DATA.mkdir(exist_ok=True)
MOIS = ["01", "02", "03"]
for mois in MOIS:
    fichier = DATA / f"yellow_tripdata_2024-{mois}.parquet"
    if not fichier.exists():
        urllib.request.urlretrieve(
            f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{mois}.parquet", fichier)
    print(fichier, f"{fichier.stat().st_size:,} octets".replace(",", " "))

archive = DATA / "NASA_access_log_Aug95.gz"
if not archive.exists():
    urllib.request.urlretrieve("https://ita.ee.lbl.gov/traces/NASA_access_log_Aug95.gz", archive)
AOUT = DATA / "NASA_access_log_Aug95"
if not AOUT.exists():
    with gzip.open(archive, "rb") as entree, open(AOUT, "wb") as sortie:
        shutil.copyfileobj(entree, sortie)
print(AOUT, f"{AOUT.stat().st_size:,} octets".replace(",", " "))

JOURS = DATA / "jours_aout"
if not JOURS.exists():
    JOURS.mkdir()
    ouverts, jour = {}, b"00"
    horodatage = re.compile(rb"\[(\d{2})/Aug/1995:")
    with open(AOUT, "rb") as entree:
        for ligne in entree:
            trouve = horodatage.search(ligne)
            if trouve:
                jour = trouve.group(1)
            if jour not in ouverts:
                ouverts[jour] = open(JOURS / f"aug95_{jour.decode()}.log", "wb")
            ouverts[jour].write(ligne)
    for f in ouverts.values():
        f.close()
print(len(list(JOURS.glob("*.log"))), "fichiers journaliers dans", JOURS)

## Partie A : organiser et déposer

### Question 1 : l'arborescence

Créez dans HDFS un répertoire par mois pour les trajets de taxis, sous la forme
`/tp2/taxis/annee=2024/mois=01`, `.../mois=02`, `.../mois=03`. Vérifiez la création avec une commande HDFS.

**Livrable** : `q1_repertoires` (liste triée des trois chemins complets, lus dans HDFS).

In [ ]:
# Question 1

### Question 2 : prévoir le nombre de blocs

Les fichiers vont être déposés avec des blocs de 16 Mo (16 777 216 octets). **Sans rien déposer**, calculez à
partir de la taille de chaque fichier local le nombre de blocs qu'il occupera dans HDFS.

**Livrable** : `q2_blocs_prevus` (dictionnaire `{"01": ..., "02": ..., "03": ...}`).

In [ ]:
# Question 2

### Question 3 : déposer et vérifier

Déposez chaque fichier dans le répertoire de son mois, avec des blocs de 16 Mo. Lisez ensuite dans HDFS le
nombre de blocs réel de chacun. Votre prévision de la question 2 est-elle juste ?

**Livrable** : `q3_blocs_reels` (dictionnaire de même forme que `q2_blocs_prevus`) et `q3_prevision_juste`
(booléen).

In [ ]:
# Question 3

### Question 4 : le dernier bloc

Quelle est la longueur, en octets, du dernier bloc du fichier de février ? Vérifiez-la par un calcul à partir
de la taille du fichier local.

**Livrable** : `q4_dernier_bloc` (entier, lu dans HDFS) et `q4_calcul_juste` (booléen : le calcul donne la
même valeur).

In [ ]:
# Question 4

## Partie B : les copies

### Question 5 : l'espace consommé

Quelle est la taille totale des trois fichiers dans `/tp2/taxis`, et quel espace occupent-ils sur les
disques des DataNodes, copies comprises ? L'espace est-il égal à la taille multipliée par le facteur de
réplication par défaut ?

**Livrable** : `q5_taille`, `q5_espace` (entiers, en octets) et `q5_egal_taille_x_3` (booléen).

In [ ]:
# Question 5

### Question 6 : moins de copies

Le fichier de mars peut être recalculé à tout moment : ramenez-le à 2 copies, en attendant la fin de
l'opération. Quel espace consomme maintenant `/tp2/taxis` ?

**Livrable** : `q6_espace` (entier, en octets, lu dans HDFS) et `q6_espace_attendu` (entier, calculé par vous
à partir des tailles et des facteurs de réplication).

In [ ]:
# Question 6

### Question 7 : où sont les copies

Le premier bloc du fichier de mars a 2 copies. Sur quels DataNodes sont-elles, et lequel des trois n'en porte
pas ? Utilisez les noms des travailleurs (`worker1`, `worker2`, `worker3`), pas les adresses IP.

**Livrable** : `q7_noeuds` (liste triée des deux noms) et `q7_absent` (nom du troisième).

In [ ]:
# Question 7

## Partie C : récupérer et vérifier

### Question 8 : récupérer un fichier

Récupérez le fichier de janvier depuis HDFS dans `data/retour/` (sur votre machine). Le fichier récupéré
est-il identique à l'original, octet pour octet ? Comparez les empreintes MD5.

**Livrable** : `q8_md5` (chaîne, empreinte du fichier récupéré) et `q8_identique` (booléen).

In [ ]:
# Question 8

## Partie D : une panne

### Question 9 : arrêter un DataNode

Arrêtez le travailleur `worker1` (`docker_compose("stop", "worker1")`). Attendez que le NameNode le déclare
mort, puis que HDFS ait fini de recréer les copies qu'il peut recréer (une minute environ). Pendant la panne,
relisez le fichier de janvier depuis HDFS : son MD5 est-il celui de l'original ? Combien de blocs de
`/tp2/taxis` restent sous-répliqués ?

**Livrable** : `q9_vivants` (entier, DataNodes vivants), `q9_md5_identique` (booléen) et
`q9_sous_repliques` (entier, blocs sous-répliqués de `/tp2/taxis` une fois HDFS stabilisé).

**Attention** : à la question 10, vous devez relancer `worker1`.

In [ ]:
# Question 9

### Question 10 : le retour du DataNode

Relancez `worker1`. Attendez qu'il soit de nouveau vivant et que HDFS ne signale plus de bloc sous-répliqué
dans `/tp2/taxis`.

**Livrable** : `q10_vivants` (entier) et `q10_sous_repliques` (entier).

In [ ]:
# Question 10

## Partie E : les petits fichiers

### Question 11 : un fichier ou beaucoup

Les logs d'août sont disponibles en un seul fichier (`data/NASA_access_log_Aug95`) et en fichiers
journaliers (`data/jours_aout/`). Déposez-les dans HDFS, avec les blocs de 128 Mo par défaut : le fichier
unique dans `/tp2/aout/mois`, les fichiers journaliers dans `/tp2/aout/jours`. Pour chacun des deux
répertoires, comptez les objets que le NameNode doit suivre : fichiers plus blocs.

**Livrable** : `q11_objets_mois` et `q11_objets_jours` (entiers), et `q11_espace_egal` (booléen : l'espace
disque consommé est-il le même dans les deux cas ?).

In [ ]:
# Question 11

## Bonus

### Question 12 (bonus) : regrouper sans passer par votre disque

Fusionnez les fichiers journaliers de `/tp2/aout/jours` en un seul fichier HDFS, `/tp2/aout/fusion/aout95.log`,
sans les copier sur votre machine. Le fichier obtenu est-il identique, octet pour octet, à l'original
`data/NASA_access_log_Aug95` ?

**Livrable** : `q12_identique` (booléen).

In [ ]:
# Question 12

### Question 13 (bonus) : HDFS par HTTP

Le NameNode expose HDFS par HTTP (WebHDFS), sur `http://localhost:9870/webhdfs/v1/<chemin>?op=...`. Sans
utiliser la commande `hdfs`, lisez dans cette interface la taille de bloc et le facteur de réplication du
fichier de février.

**Livrable** : `q13_taille_bloc` et `q13_replication` (entiers).

In [ ]:
# Question 13

## Fin du TP

Vérifiez que les trois travailleurs sont vivants (`docker compose ps`, onglet Datanodes du NameNode). Si vous
avez fini avant la fin du créneau, parcourez dans le NameNode (Utilities > Browse the file system) les
répertoires `/tp2/taxis` et `/tp2/aout` : repérez, pour chaque fichier, sa taille de bloc et sa réplication.